In [7]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

text = "All right, so here we are, in front of the elephants. Cool thing about these guys is that they have really long trunks. And that's cool. And that's pretty much all there is to say."

sentences = sent_tokenize(text)

for i, s in enumerate(sentences):
    print(f"Sentence {i}: {s}")

Sentence 0: All right, so here we are, in front of the elephants.
Sentence 1: Cool thing about these guys is that they have really long trunks.
Sentence 2: And that's cool.
Sentence 3: And that's pretty much all there is to say.


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/diegonunez/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [8]:
# Simulating real VibeVoice output — multiple segments with timestamps and speakers
segments = [
    {
        "start": 0.0,
        "end": 14.0,
        "speaker": 0,
        "content": "All right, so here we are, in front of the elephants. Cool thing about these guys is that they have really long trunks. And that's cool."
    },
    {
        "start": 14.0,
        "end": 16.2,
        "speaker": None,
        "content": "[Unintelligible Speech]"
    },
    {
        "start": 16.2,
        "end": 19.01,
        "speaker": 0,
        "content": "And that's pretty much all there is to say."
    }
]

# Split each segment into sentences, keeping the timestamp and speaker info
all_sentences = []

for seg in segments:
    # Skip non-speech segments like [Music] or [Unintelligible Speech]
    if seg["content"].startswith("["):
        continue
    
    # Split the segment's content into individual sentences
    sentences = sent_tokenize(seg["content"])
    
    # For each sentence, estimate its timestamp
    # We spread the segment's time range evenly across its sentences
    seg_duration = seg["end"] - seg["start"]
    time_per_sentence = seg_duration / len(sentences) if sentences else 0
    
    for i, sentence in enumerate(sentences):
        all_sentences.append({
            "text": sentence,
            "start": round(seg["start"] + (i * time_per_sentence), 2),
            "end": round(seg["start"] + ((i + 1) * time_per_sentence), 2),
            "speaker": seg["speaker"],
        })

# See the result
for s in all_sentences:
    print(f"[{s['start']:.1f}s - {s['end']:.1f}s] Speaker {s['speaker']}: {s['text']}")

[0.0s - 4.7s] Speaker 0: All right, so here we are, in front of the elephants.
[4.7s - 9.3s] Speaker 0: Cool thing about these guys is that they have really long trunks.
[9.3s - 14.0s] Speaker 0: And that's cool.
[16.2s - 19.0s] Speaker 0: And that's pretty much all there is to say.


In [9]:
# First install sentence-transformers
!pip install sentence-transformers


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [10]:
from sentence_transformers import SentenceTransformer

# Load the model — this is the one from your proposal
model = SentenceTransformer("all-MiniLM-L6-v2")

# Get just the text from our sentences
texts = [s["text"] for s in all_sentences]

# Convert each sentence into a 384-dimensional vector
embeddings = model.encode(texts)

# Let's see what we got
print(f"Number of sentences: {len(embeddings)}")
print(f"Vector size: {embeddings[0].shape}")
print(f"\nFirst sentence: '{texts[0]}'")
print(f"Its vector (first 10 numbers): {embeddings[0][:10]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of sentences: 4
Vector size: (384,)

First sentence: 'All right, so here we are, in front of the elephants.'
Its vector (first 10 numbers): [ 0.02223937  0.04814821  0.05367174  0.02535025  0.04559984  0.01964867
 -0.00679244 -0.0578038   0.06792793  0.02930185]


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

# Compare each sentence with the next one
print("Cosine similarity between consecutive sentences:\n")

for i in range(len(embeddings) - 1):
    # cosine_similarity expects 2D arrays, so we reshape with [i:i+1]
    score = cosine_similarity(
        embeddings[i:i+1],    # sentence i
        embeddings[i+1:i+2]   # sentence i+1
    )[0][0]
    
    print(f"  Sentence {i} vs {i+1}: {score:.3f}")
    print(f"    '{texts[i][:50]}...'")
    print(f"    '{texts[i+1][:50]}...'\n")

Cosine similarity between consecutive sentences:

  Sentence 0 vs 1: 0.158
    'All right, so here we are, in front of the elephan...'
    'Cool thing about these guys is that they have real...'

  Sentence 1 vs 2: 0.202
    'Cool thing about these guys is that they have real...'
    'And that's cool....'

  Sentence 2 vs 3: 0.402
    'And that's cool....'
    'And that's pretty much all there is to say....'



In [12]:
# A longer transcript with clear topic changes
lecture_sentences = [
    # Topic 1: What are neural networks
    "Today we're going to talk about neural networks.",
    "A neural network is a computing system inspired by the brain.",
    "It consists of layers of interconnected nodes called neurons.",
    "Each connection between neurons has a weight that gets adjusted during training.",
    
    # Topic 2: Training process
    "Now let's discuss how we actually train these networks.",
    "Training involves feeding data through the network and comparing the output to the expected result.",
    "The difference between the output and the expected result is called the loss.",
    "We use an algorithm called backpropagation to minimize this loss.",
    "Backpropagation computes gradients that tell us how to adjust each weight.",
    
    # Topic 3: Practical tips
    "Let me share some practical tips for training your models.",
    "Always start with a small learning rate and increase it gradually.",
    "Make sure to split your data into training and validation sets.",
    "Overfitting is a common problem where the model memorizes the training data.",
    "You can use techniques like dropout and data augmentation to prevent overfitting.",
]

# Embed all sentences
lecture_embeddings = model.encode(lecture_sentences)

# Compute consecutive similarities
similarities = []
for i in range(len(lecture_embeddings) - 1):
    score = cosine_similarity(
        lecture_embeddings[i:i+1],
        lecture_embeddings[i+1:i+2]
    )[0][0]
    similarities.append(score)

# Display results
print("Consecutive similarity scores:\n")
for i, score in enumerate(similarities):
    marker = " ◄── low" if score < 0.3 else ""
    print(f"  {i} vs {i+1}: {score:.3f}  {marker}")
    print(f"    '{lecture_sentences[i][:60]}'")
    print(f"    '{lecture_sentences[i+1][:60]}'\n")

Consecutive similarity scores:

  0 vs 1: 0.658  
    'Today we're going to talk about neural networks.'
    'A neural network is a computing system inspired by the brain'

  1 vs 2: 0.676  
    'A neural network is a computing system inspired by the brain'
    'It consists of layers of interconnected nodes called neurons'

  2 vs 3: 0.521  
    'It consists of layers of interconnected nodes called neurons'
    'Each connection between neurons has a weight that gets adjus'

  3 vs 4: 0.407  
    'Each connection between neurons has a weight that gets adjus'
    'Now let's discuss how we actually train these networks.'

  4 vs 5: 0.584  
    'Now let's discuss how we actually train these networks.'
    'Training involves feeding data through the network and compa'

  5 vs 6: 0.415  
    'Training involves feeding data through the network and compa'
    'The difference between the output and the expected result is'

  6 vs 7: 0.404  
    'The difference between the output and the expecte

In [13]:
import numpy as np

# Find the cutoff — bottom 15% of similarity scores
cutoff = np.percentile(similarities, 15)
print(f"All scores: {[round(s, 3) for s in similarities]}")
print(f"15th percentile cutoff: {cutoff:.3f}")
print(f"\nBoundaries detected (scores below {cutoff:.3f}):\n")

# Find where topic changes happen
boundaries = []
for i, score in enumerate(similarities):
    if score < cutoff:
        boundaries.append(i + 1)  # boundary is AFTER sentence i
        print(f"  Topic change after sentence {i}: score {score:.3f}")
        print(f"    '{lecture_sentences[i][:60]}'")
        print(f"    '{lecture_sentences[i+1][:60]}'\n")

print(f"\nBoundary positions: {boundaries}")
print(f"This splits the transcript into {len(boundaries) + 1} topics")

All scores: [np.float32(0.658), np.float32(0.676), np.float32(0.521), np.float32(0.407), np.float32(0.584), np.float32(0.415), np.float32(0.404), np.float32(0.545), np.float32(0.266), np.float32(0.416), np.float32(0.483), np.float32(0.3), np.float32(0.62)]
15th percentile cutoff: 0.383

Boundaries detected (scores below 0.383):

  Topic change after sentence 8: score 0.266
    'Backpropagation computes gradients that tell us how to adjus'
    'Let me share some practical tips for training your models.'

  Topic change after sentence 11: score 0.300
    'Make sure to split your data into training and validation se'
    'Overfitting is a common problem where the model memorizes th'


Boundary positions: [9, 12]
This splits the transcript into 3 topics


In [14]:
# Group sentences into topics based on boundary positions
topics = []
start_idx = 0

for boundary in boundaries:
    topics.append(lecture_sentences[start_idx:boundary])
    start_idx = boundary

# Don't forget the last topic (after the final boundary)
topics.append(lecture_sentences[start_idx:])

# Display the topics
for i, topic in enumerate(topics):
    print(f"{'='*60}")
    print(f"TOPIC {i + 1} ({len(topic)} sentences)")
    print(f"{'='*60}")
    for sentence in topic:
        print(f"  {sentence}")
    print()

TOPIC 1 (9 sentences)
  Today we're going to talk about neural networks.
  A neural network is a computing system inspired by the brain.
  It consists of layers of interconnected nodes called neurons.
  Each connection between neurons has a weight that gets adjusted during training.
  Now let's discuss how we actually train these networks.
  Training involves feeding data through the network and comparing the output to the expected result.
  The difference between the output and the expected result is called the loss.
  We use an algorithm called backpropagation to minimize this loss.
  Backpropagation computes gradients that tell us how to adjust each weight.

TOPIC 2 (3 sentences)
  Let me share some practical tips for training your models.
  Always start with a small learning rate and increase it gradually.
  Make sure to split your data into training and validation sets.

TOPIC 3 (2 sentences)
  Overfitting is a common problem where the model memorizes the training data.
  You can 

In [15]:
# Try different percentile values and see how it changes the segmentation
for pct in [10, 15, 20, 25, 30]:
    cutoff = np.percentile(similarities, pct)
    num_boundaries = sum(1 for s in similarities if s < cutoff)
    print(f"  Percentile {pct}% → cutoff {cutoff:.3f} → {num_boundaries + 1} topics")

  Percentile 10% → cutoff 0.321 → 3 topics
  Percentile 15% → cutoff 0.383 → 3 topics
  Percentile 20% → cutoff 0.405 → 4 topics
  Percentile 25% → cutoff 0.407 → 4 topics
  Percentile 30% → cutoff 0.412 → 5 topics
